# Distributed Systems Theory

## What's covered

- **Why distributed systems are different** — the assumptions that break and the new failures that show up
- **The CAP theorem** — what it actually says, what it doesn't, and the PACELC refinement
- **Consistency models** — linearizable, sequential, causal, eventual, and where real systems land
- **Clocks** — why wall-clock time isn't trustworthy and how Lamport and vector clocks fix it
- **Consensus** — Raft and Paxos at a conceptual level, and what they let you build
- **Leader election** — the foundation under failover, sharding, and locking


## Why distributed is different

A single-process program has a comfortable property: **everything either happened or didn't.** A function call returns or throws. State is shared coherently in one address space. The CPU's clock advances monotonically. The world is *consistent*.

The moment you add a network, all of those properties weaken. A request you sent might have succeeded or failed or be still in flight — you don't know. State held in one process can diverge from state held in another. Two clocks on two machines don't agree. And — uniquely to distributed systems — you can have **partial failure**: half the system works, half doesn't, and the half that works has to keep going somehow.

Peter Deutsch's **eight fallacies of distributed computing** name the assumptions that quietly poison single-machine reasoning when applied to distributed systems:

1. The network is reliable.
2. Latency is zero.
3. Bandwidth is infinite.
4. The network is secure.
5. Topology doesn't change.
6. There is one administrator.
7. Transport cost is zero.
8. The network is homogeneous.

Each of these is *almost* true on a laptop and *plainly* false across a fleet of machines. The rest of this notebook is, in one form or another, vocabulary for reasoning about the failure modes those broken assumptions introduce.

**Three categories of failures** to keep in mind:

- **Crash failures** — a node stops responding. Easy to detect (timeouts), easy to reason about. Most real failures are this.
- **Omission failures** — a node responds to some requests but not others; messages get dropped silently. Harder.
- **Byzantine failures** — a node sends *wrong* answers (corrupted, malicious, buggy). Rare in trusted environments. Common in adversarial ones (cryptocurrencies, public-internet protocols).

Most distributed-systems work assumes crash failures. Byzantine fault tolerance is a much harder problem with much higher overhead.


## The CAP theorem — what it actually says

Eric Brewer's conjecture, later proved by Gilbert and Lynch, is the most-cited and most-misunderstood result in distributed systems. The precise statement:

> In the presence of a **network partition**, a distributed data store must choose between **consistency** and **availability**.

The misunderstanding is to read it as "pick two of {C, A, P}". You don't choose P. **Partitions happen**, whether you want them or not. The real choice the theorem describes is what to do *during* a partition.

**During a partition, you have two options:**

- **CP** — favor consistency. If a node can't reach a quorum, it refuses requests. No stale reads, no conflicting writes — but the system is unavailable from that node's perspective. ZooKeeper, etcd, Spanner, most relational databases under network failure.
- **AP** — favor availability. Each side of the partition keeps accepting requests, returning whatever it knows locally. The system stays up — but it may serve stale data, and the two sides will diverge until reconciled. Cassandra, DynamoDB, Riak.

**The C in CAP is a specific consistency** — *linearizability*, the strongest possible. It means every read sees the most recent committed write. Weaker consistency models (eventual, causal) are still meaningful during partitions; they just aren't what CAP's "C" measures.

The real-world implication is more subtle than "pick CP or AP." Most systems are CP for some operations and AP for others — a database that's strongly consistent for transactional writes might be eventually consistent for read replicas. The decision is per-workload, not per-system.


## Beyond CAP — PACELC

CAP describes behaviour during a network partition. But partitions are rare. The more interesting question for most of the time is: **what trade-off does the system make when there isn't a partition?**

Daniel Abadi's **PACELC** is the refinement: *if Partition, then trade Availability or Consistency (the CAP choice); Else (no partition), trade Latency or Consistency.*

```
   Partition?                  ELSE
       |                        |
       |---PA---availability    |---EL---low latency
       |                        |
       |---PC---consistency     |---EC---strong consistency
```

A system can be classified as PA/EL, PA/EC, PC/EL, or PC/EC. A few examples:

- **DynamoDB:** PA/EL — during a partition it prefers availability; even in normal operation it prefers low latency over strong consistency.
- **Spanner:** PC/EC — even during a partition it refuses to serve from a minority; in normal operation it gives you strong consistency, accepting the latency cost of cross-region consensus.
- **MongoDB (with defaults):** PA/EC — available during partition, strongly consistent (linearizable) reads from primary in normal operation.

PACELC is more useful than CAP for actual decisions because the partition case is the *exception*; the steady-state trade-off (the ELC part) is what you live with every day.


## Consistency models — the spectrum

"Consistent" by itself is not a measurement. There are many consistency models, ordered from strongest (and most expensive) to weakest (and cheapest). A system specifies which model it guarantees; the application has to be designed around what that model does and does not promise.

```
   strongest                                       weakest
   --------                                       --------

   linearizable  ->  sequential  ->  causal  ->  eventual
   (real-time           (some        (cause/    (will agree
    order)              order)       effect)     eventually)
```

### Linearizable consistency (strongest)

Every operation appears to take effect atomically at some point between its start and its end, and operations are ordered by real time. If write W finished before read R started, R sees W (or a later write).

This is what most people mean intuitively when they say "the system is consistent." It's also expensive — every operation must coordinate with a quorum across the cluster. ZooKeeper, etcd, Spanner offer this.

### Sequential consistency

All nodes see operations in the *same order*, but that order doesn't have to match real time. Two writes that happened concurrently can be ordered either way, as long as everyone agrees.

Weaker than linearizable: a read can return an older value as long as the order is consistent. Easier to implement. Not commonly offered as a top-line model in modern systems; it's more often a theoretical stepping-stone.

### Causal consistency

Operations that are *causally related* (B happened because of A) are seen in the same order by everyone. Concurrent operations can be seen in different orders. "If I reply to your message, every observer sees your message before my reply" — but two unrelated messages can be seen in either order.

Causal consistency captures the natural expectation for collaborative apps (chat, comments, shared documents) and is significantly cheaper than linearizable. Used by COPS, Bayou, and as a building block in modern systems.

### Eventual consistency (weakest commonly seen)

If no new updates are made, eventually all replicas converge to the same value. That's the only guarantee. In between, replicas can disagree arbitrarily. Reads can return stale values; writes can take seconds to propagate.

This is the consistency level of DynamoDB's eventually-consistent reads, Cassandra's default, the DNS system, and most CDN caches. It's enormously cheaper than stronger models and is acceptable whenever the application can tolerate a few seconds of staleness.

### Which to pick

The honest answer: **the weakest one that lets the application be correct**. Strong consistency is expensive in latency, availability, and dollars; pay for it where you need it (financial transactions, distributed locks, leader election) and use weaker models where you don't (social feeds, cached lookups, analytics).


## Clocks — why wall-clock time isn't trustworthy

Two events happened on two machines. Which came first?

The naïve answer is "look at the wall-clock timestamps." This is wrong for three reasons.

1. **Clock skew.** Two machines' clocks drift apart by milliseconds to seconds. NTP corrects this periodically, but between corrections clocks can be off by enough to invert event order. A "later" event on machine B may have a smaller timestamp than an "earlier" event on machine A.
2. **Clock jumps.** When NTP corrects a clock, it may jump forward or backward. An event with timestamp T can be followed by an event with timestamp T-5 seconds.
3. **No global clock.** Even Google's TrueTime, which uses GPS and atomic clocks, only gives bounded uncertainty — "this event happened between T1 and T2 globally." Without specialized hardware, you have nothing like that.

The fix isn't a better physical clock — it's a different *kind* of clock. **Logical clocks** measure *causality*, not real time. Two events that didn't influence each other are simply *concurrent*; the system doesn't pretend to know which came first.

### Lamport clocks

The simplest logical clock. Each node holds a counter `L`. Rules:

1. **Before any event** (local or send), increment: `L += 1`.
2. **On send**, attach the current `L` to the message.
3. **On receive** with attached `L_msg`, update: `L = max(L, L_msg) + 1`.

If event A *happened before* event B (B saw A's effect), then `L(A) < L(B)`. The reverse is not guaranteed — `L(A) < L(B)` doesn't mean A happened before B; they could be unrelated. Lamport clocks give a *partial order* on events that captures "could have caused."

### Vector clocks

A more informative logical clock. Each node `i` holds a vector `V` of counters, one per node. Rules:

1. **Before any local event**, increment own slot: `V[i] += 1`.
2. **On send**, attach the current `V` to the message.
3. **On receive** with attached `V_msg`, take pointwise max, then increment own slot.

Comparison: `V1 ≤ V2` if every slot in V1 is ≤ the corresponding slot in V2. If `V1 < V2`, event 1 happened before event 2. If neither is ≤ the other, the events are *concurrent* — and the system can detect that, unlike with Lamport clocks.

Vector clocks are the basis of how Dynamo-style systems detect and reconcile conflicting concurrent writes.


### Lamport and vector clocks in code


In [ ]:
# Lamport clocks — partial order capturing "happened-before".

class LamportClock:
    def __init__(self):
        self.t = 0

    def tick(self) -> int:           # before any local event
        self.t += 1
        return self.t

    def send(self) -> int:           # attach this to outgoing message
        return self.tick()

    def receive(self, msg_t: int) -> int:
        self.t = max(self.t, msg_t) + 1
        return self.t


# Three nodes; A sends to B, B sends to C.
A, B, C = LamportClock(), LamportClock(), LamportClock()

t1 = A.send();             print(f"A sends:    A.t={t1}")
t2 = B.receive(t1);        print(f"B receives: B.t={t2}")
t3 = B.send();             print(f"B sends:    B.t={t3}")
t4 = C.receive(t3);        print(f"C receives: C.t={t4}")
# Note: A's send (1) < B's receive (2) < B's send (3) < C's receive (4).
# The total order respects causality.


# Vector clocks — full happened-before relation, can detect concurrency.

class VectorClock:
    def __init__(self, node_id: str, all_nodes: list[str]):
        self.id = node_id
        self.v = {n: 0 for n in all_nodes}

    def tick(self) -> dict[str, int]:
        self.v[self.id] += 1
        return dict(self.v)

    def send(self) -> dict[str, int]:
        return self.tick()

    def receive(self, msg_v: dict[str, int]) -> dict[str, int]:
        for n, t in msg_v.items():
            self.v[n] = max(self.v[n], t)
        self.v[self.id] += 1
        return dict(self.v)


def happens_before(v1, v2) -> bool:
    return all(v1[n] <= v2[n] for n in v1) and any(v1[n] < v2[n] for n in v1)


def concurrent(v1, v2) -> bool:
    return not happens_before(v1, v2) and not happens_before(v2, v1)


nodes = ["A", "B", "C"]
A, B, C = (VectorClock(n, nodes) for n in nodes)

a1 = A.send();             print(f"A sends:    {a1}")
b1 = B.receive(a1);        print(f"B receives: {b1}")
# Now an independent event on C — concurrent with A->B.
c1 = C.tick();             print(f"C ticks:    {c1}")

print(f"\nA's send vs C's tick:")
print(f"  happens-before(A, C) = {happens_before(a1, c1)}")
print(f"  happens-before(C, A) = {happens_before(c1, a1)}")
print(f"  concurrent           = {concurrent(a1, c1)}")
print(f"\nA's send vs B's receive (causal chain):")
print(f"  happens-before(A, B) = {happens_before(a1, b1)}")


**The big picture.** Lamport clocks give you a single number you can compare — useful as a total order in logs and audit trails — but they can't tell you whether two events were concurrent. Vector clocks can, at the cost of carrying one number per node in every message. Most production systems use Lamport-style hybrid logical clocks (HLC) that combine physical time with a logical counter, getting the best of both for typical workloads.


## Consensus — making nodes agree

The most fundamental problem in distributed systems: **how do N nodes agree on a single value, given that some might fail and the network might drop messages?**

This shows up everywhere. Choosing a new leader after the old one crashes. Committing a transaction across replicas. Adding a new node to a cluster. Updating a configuration that everyone needs to see consistently. If you can solve consensus once, you can build almost every distributed-systems primitive on top of it.

The FLP impossibility result (Fischer-Lynch-Paterson, 1985) proved that **no deterministic consensus protocol can guarantee both safety and liveness in an asynchronous network with even one possible crash failure.** Real-world protocols therefore make assumptions — typically that the network is *partially synchronous* (messages arrive eventually, even if late) and that a majority of nodes are up. Under those assumptions, consensus is achievable.

Two protocols dominate the modern landscape: **Paxos** and **Raft**.

### Raft — designed to be understandable

Raft was published in 2014 explicitly as "Paxos but understandable." It's now the dominant choice for new systems. Three pieces:

**1. Leader election.** Nodes are in one of three states: *follower*, *candidate*, *leader*. When a follower stops hearing from a leader, it becomes a candidate, increments its *term* (a monotonic election counter), and requests votes. A candidate that gets a majority becomes the leader for that term.

**2. Log replication.** All writes go to the leader. The leader appends to its local log, then sends the entry to followers. Once a majority have written it to *their* logs, the entry is **committed** and applied to state machines.

**3. Safety properties.** Two crucial invariants:

- **Leader completeness** — the elected leader for term T has every entry committed in any earlier term. Achieved by only voting for candidates whose log is at least as up-to-date as the voter's.
- **Log matching** — if two logs have the same entry at the same index, all entries before that index are identical.

These invariants combined mean Raft never loses committed data even under arbitrary leader failures.

### Paxos — the original

Lamport's Paxos is older (1989, formally published 1998) and has the same guarantees as Raft. The reason Raft exists is that Paxos is famously hard to understand and implement correctly — the original paper required two follow-up papers to clarify it, and a generation of production Paxos implementations had subtle correctness bugs.

In modern systems Paxos still appears (Spanner, Chubby) but Raft is dominant for new work. They are equivalent in capability; pick based on what your team can implement and operate confidently.

### What you build on top

Once you have consensus, you get:

- **Replicated state machines** — every node runs the same operations in the same order, so they all reach the same state.
- **Distributed locks** — a leader election is itself a distributed lock; etcd and ZooKeeper expose this directly.
- **Atomic configuration changes** — Kubernetes uses etcd (Raft) for every cluster-state change.
- **Strongly consistent metadata** — service discovery, leader registries, feature flags.

Consensus is expensive — every commit requires a round-trip to a majority. You don't run *application* writes through Raft; you run *coordination* writes through it. The application typically uses Raft to elect a leader, then the leader serves application traffic at much higher throughput than raw Raft could.


## Leader election — the foundation under failover

Many of the patterns in the rest of the series — primary failover in databases, single-writer guarantees in queues, exclusive ownership of shards — reduce to "one and only one node is the leader at any time." That requires **leader election**.

**The hard part is preventing two leaders.** A *split-brain* — two nodes both convinced they're the leader — is one of the most damaging failure modes possible. Both write conflicting updates. Both serve clients. Reconciliation is sometimes impossible without data loss.

Three mechanisms, ordered by how reliably they prevent split-brain:

- **Consensus-based** (Raft/Paxos/etcd/ZooKeeper) — the leader is elected by majority vote with monotonic terms. A new election always invalidates the previous leader's term. **Safe under all failure modes.** This is the modern default; if you need leader election, use a system that provides it.
- **Lease-based with a strongly consistent store** — leaders hold a time-limited lease in a database/etcd that they renew. If they fail to renew, the lease expires and someone else can take it. Safe if and only if the store providing leases is strongly consistent and the timing assumptions hold.
- **Heartbeat-based with hand-rolled election** — nodes send heartbeats; on missed heartbeats, the survivors hold an ad-hoc election among themselves. **Fragile.** Network partitions create dual leaders; deploys race the election; correctness depends on synchronized clocks. Discouraged in modern designs.

**Fencing tokens** are a critical defence even with consensus. Every leader is issued a monotonically increasing token. Downstream systems (databases, locks, file systems) record the highest token they've seen and *reject* operations with older tokens. This guarantees that a deposed leader who didn't realize it lost the lease can't continue to write — its token is stale. Without fencing tokens, even consensus-based leader election can let zombie leaders cause damage.

The pragmatic recipe for any system needing a leader: **use etcd or ZooKeeper for the election, and use fencing tokens on every operation the leader performs.** Don't roll your own.


Notebook five turns to **Messaging & Event-Driven Architectures** — the asynchronous half of distributed systems. Message queues and their delivery guarantees. Publish-subscribe. Event-driven architecture as a system shape. Event sourcing as a way to make state derivable from a log. And command query responsibility segregation as a split-read-and-write pattern. The consistency and consensus vocabulary from this notebook is the foundation those patterns build on.
